# 06. Compare Random Forest and JEPA on the same held-out sources

This capstone connects original video, shared pose extraction, features, learned representations, and source-level evaluation. It reads actual experiment artifacts when available and otherwise gives executable commands. It never replaces missing real results with synthetic accuracy.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Fix the comparison before looking at test scores

Both classifiers must receive identical accepted windows from identical sources. The RF uses measured joint-angle and motion summaries. JEPA uses an encoder representation with a training-only standardized logistic probe. Data splits, pose failures, quality rules, and target labels are shared. Hyperparameter selection uses validation sources with a declared candidate budget.

Exp5 demonstrates the classical feature route. Historical scores from a different cohort are not a controlled baseline. This project retrains the classical model on the new five-label cohort.

In [ ]:
display(SVG(filename=str(PROJECT / "images/comparison.svg")))

In [ ]:
from worldmodels.data import build_inventory, assign_splits
inventory, audit = build_inventory(REPOSITORY)
partition = assign_splits(inventory, seed=27)
if REAL:
    _, _, clean_metadata = real_inputs()
    display(clean_metadata.groupby(["split","label"]).source_id.nunique().unstack())
else:
    display(partition.loc[partition.eligible].groupby(["split","label"]).source_id.nunique().unstack())

## Step 2. Understand the 37 observed features

The first 15 follow the core exp5 construction: six mean hip/knee/ankle angles, three bilateral differences, and six angle ranges. The extension adds angle standard deviations, elbow summaries, foot-axis inclination, wrist motion, and ankle separation. Shared Ambient geometry computes lower-limb angles. Missing observations remain missing, and training-only imputation handles incomplete feature vectors.

The feature input is the same normalized 16-joint window used by JEPA. No source identifier, class label, or filename becomes a predictor. Units are projected angles and torso-normalized movement, not measured 3D anatomy or physical gait speed.

In [ ]:
from worldmodels.notebook_support import lesson_data
from worldmodels.features import build_features
fixture = lesson_data(n=4)
feature_matrix, feature_names = build_features(fixture["x"], fixture["valid"], fps=fixture["metadata"].sample_fps.to_numpy() if REAL else 15)
print(EVIDENCE_LABEL + " feature matrix:", feature_matrix.shape)
display(pd.DataFrame({"column": np.arange(len(feature_names)), "name": feature_names}))

## Step 3. Prepare and run a real bounded pilot

Run these commands from the experiment directory. The clean builder requires the previously extracted real 16-frame pose caches and the original `extended-input-v1` cohort. It verifies provenance, reapplies the person gate using fresh GAVD bounding-box annotations, and excludes the complete sources `CMls0LwK5OU` from validation and `eBvzFkcvScg` from training. It creates `pilot-input-clean-v1` and `extended-input-clean-v1` without overwriting the earlier cohorts. Source assignments remain fixed. This is an intentionally short classifier pilot, distinct from the longer forecasting study. If the clean inputs already exist, reuse their verified artifacts instead of rerunning the immutable builder.

```bash
uv run python -m worldmodels.clean_data
uv run python -m worldmodels.workflow compare --input artifacts/runs/pilot-input-clean-v1 --output artifacts/runs/pilot-comparison-clean-v1 --epochs 5
```

The person-gate ledger is `docs/source_exclusions.json`. The longer clean forecasting cohort has its own preparation identity and reuse certificate. A new temporal horizon requires a new declared cohort and renewed person/quality checks; a raw `workflow prepare` call alone does not establish person eligibility. Output directories refuse accidental overwrite.

In [ ]:
input_dir = prepared_path()
result_dir = comparison_path()
if (input_dir / "preparation.json").exists():
    import json
    display(pd.Series(json.loads((input_dir / "preparation.json").read_text())))
else:
    print("No prepared real pilot exists yet. Run the preparation command above.")

## Step 4. Inspect attrition before accuracy

A model comparison becomes misleading if one method quietly excludes difficult sources. The runner loads one prepared matrix for every approach. It requires all five labels in every split after extraction and refuses to proceed if a class disappears. Inspect the remaining number of sources, not only the number of windows.

In [ ]:
if (input_dir / "windows.csv").exists():
    window_metadata = pd.read_csv(input_dir / "windows.csv")
    display(window_metadata.groupby(["split","label"]).agg(windows=("label","size"),sources=("source_id","nunique")))
    if (input_dir / "attrition.csv").exists():
        attrition = pd.read_csv(input_dir / "attrition.csv", keep_default_na=False)
        display(attrition.loc[attrition.exclusion_reason != ""].head(20))
else:
    print("Pose attrition is unavailable until preparation completes.")

## Step 5. Read actual source-level results

During development, `WM_RELEASE_TEST_RESULTS=0` displays validation selections and source support only. Final test tables require an explicit release after the revision protocol is frozen. Training-source input-video inference remains available before release.

Predictions are averaged within sequence and then within source. The primary comparison is binary balanced accuracy at the fixed supervised probability threshold of 0.5. Five-class macro-F1, AUROC, average precision, sensitivity, specificity, and per-class recall provide complementary views.

A paired source bootstrap resamples identical held-out source indexes for both methods. With very few sources in a condition, the interval can still underrepresent population uncertainty. It does not measure training-seed variability or external clinical generalization.

In [ ]:
import json
results_path = result_dir / "results.json"
raw_result_record = json.loads(results_path.read_text()) if results_path.exists() else None
results = raw_result_record
if results is not None and not RELEASE_TEST_RESULTS:
    permitted = ("identity", "status", "support", "validation_selection", "representation_diagnostics", "limitations", "seed", "n_windows")
    results = {key: raw_result_record[key] for key in permitted if key in raw_result_record}
    display(pd.DataFrame(results.get("validation_selection", [])))
    display(pd.DataFrame(results.get("support", [])))
    print("Final test metrics are sealed. Set WM_RELEASE_TEST_RESULTS=1 only after the revision protocol is frozen.")
if results is not None and RELEASE_TEST_RESULTS:
    score_columns = ["binary_balanced_accuracy", "macro_f1", "abnormal_auroc", "sensitivity", "specificity"]
    display(pd.DataFrame(results["classifiers"]).T.reindex(columns=score_columns))
    display(pd.Series(results["paired_jepa_minus_rf"]))
else:
    print("No test metric is displayed while results are absent or the test-release gate is closed.")

## Step 6. Read the error structure and reference detector separately

The supervised classifier predicts labels after training on the selected conditions. The anomaly branch is trained only on normal sources and scores future prediction errors. Their thresholds and intended uses differ. Preserve the normal checkpoint when adapting a separate classification encoder.

A confusion matrix can reveal a model that predicts the majority condition, but it cannot explain disease mechanisms. Inspect original video and pose visibility for misclassified sources. Report the one-source cerebral-palsy test limitation and incomplete person identity review with every pilot result.

In [ ]:
if results is not None and RELEASE_TEST_RESULTS:
    chosen = results["classifiers"]["jepa_adapted"]
    display(pd.DataFrame(chosen["confusion_matrix"], index=chosen["classes"], columns=chosen["classes"]))
    display(pd.Series(results["anomaly"]))
    display(Markdown("**Recorded limitations:**"))
    display(pd.Series(results["limitations"]))
else:
    print("Test predictions and anomaly test metrics remain withheld until WM_RELEASE_TEST_RESULTS=1.")

## Step 7. Classify an input walking interval

The inference adapter takes an existing video and a declared walking interval without requiring an input condition label. It applies the same pose/window contract, loads the fitted RF and JEPA probes, and reports sequence probabilities plus normal-reference forecast error. The interval should contain the intended walking person. A predicted condition remains a dataset-label prediction rather than a diagnosis.

After the real comparison has completed, the command-line interface is:

```bash
uv run python -m worldmodels.inference --video /absolute/path/to/walking.mp4 --run artifacts/runs/pilot-comparison-clean-v1 --start-frame 1 --end-frame 96 --pose-model ../../data/models/pose_landmarker_lite.task
```

Real mode always runs this actual input-video inference against the fitted clean classifiers. Synthetic mode runs it only when `WM_RUN_INFERENCE=1`. No condition label is passed into the classifier.

In [ ]:
if REAL or os.getenv("WM_RUN_INFERENCE", "0") == "1":
    from worldmodels.inference import classify_video
    from worldmodels.data import find_video
    sample = real_example() if REAL else partition.loc[partition.eligible & (partition.label == "normal")].iloc[0]
    source_video = find_video(REPOSITORY, sample.source_id, sample.label)
    report = classify_video(source_video, result_dir,
                            REPOSITORY / "data/models/pose_landmarker_lite.task",
                            start_frame=int(sample.first_frame), end_frame=min(int(sample.last_frame), int(sample.first_frame)+95))
    if REAL and not report.get("classifiers"):
        raise RuntimeError("Real input-video inference produced no classifier predictions")
    display(report)
else:
    print("Input-video inference is available after a fitted comparison: set WM_RUN_INFERENCE=1.")

### Predict normal versus abnormal for that same walking interval

The five-class probabilities above describe the condition-label task. The direct binary classifiers below are separately fitted frozen readouts for normal versus abnormal. They use the same input video interval and the same frozen feature/encoder contracts. This call applies the actual Random Forest and primary JEPA binary classifiers; it does not derive its prediction by summing the five-class probabilities.

The input example belongs to a training source for this tutorial demonstration, so its prediction is not a held-out performance estimate. The reported probabilities are research outputs for dataset annotations, not a clinical diagnosis. No truth label is passed to either video-inference function.

In [ ]:
if REAL or os.getenv("WM_RUN_INFERENCE", "0") == "1":
    from worldmodels.binary_inference import classify_video as classify_binary_video
    binary_development = Path(os.getenv("WM_BINARY_DEVELOPMENT") or PROJECT / "artifacts/runs/binary-development-clean-v1")
    binary_video_report = classify_binary_video(
        source_video, binary_development, REPOSITORY / "data/models/pose_landmarker_lite.task",
        start_frame=int(sample.first_frame),
        end_frame=min(int(sample.last_frame), int(sample.first_frame)+95))
    if not binary_video_report.get("classifiers"):
        raise RuntimeError("Actual direct-binary video inference returned no predictions")
    binary_video_rows = []
    for name, result in binary_video_report["classifiers"].items():
        binary_video_rows.append({"model": name, **result["probabilities"],
                                  "predicted_annotation": result["predicted_annotation"],
                                  "threshold": result["threshold"]})
    display(pd.DataFrame(binary_video_rows))
    report["direct_binary"] = binary_video_report
    binary_video_output = lesson_directory("06_capstone_rf_vs_jepa") / "binary_video_report.json"
    binary_video_output.write_text(json.dumps(binary_video_report, indent=2))
else:
    print("Direct-binary input-video inference runs in real mode or with WM_RUN_INFERENCE=1.")

## Step 8. Compare the direct binary endpoint

A five-class classifier and a direct normal-versus-abnormal classifier optimize different targets. Deriving abnormal probability by adding four condition probabilities can behave differently from fitting the binary endpoint directly, especially when class weighting changes the effective training distribution. The revision fixes the binary target and chooses its probe settings using validation sources before releasing the final test table.

The table below reads the actual `binary-test-clean-v1` result. It includes the matched Random Forest and JEPA probes, identifies the validation-selected primary JEPA representation, and reports its paired comparison with RF. It does not infer an improvement merely because the endpoint changed. This remains a small source-grouped study without external clinical validation.

In [ ]:
if REAL and RELEASE_TEST_RESULTS:
    binary_path = Path(os.getenv("WM_BINARY_TEST") or PROJECT / "artifacts/runs/binary-test-clean-v1/results.json")
    if not binary_path.is_file():
        raise FileNotFoundError("Released capstone requires the completed direct-binary test record.")
    binary_report = json.loads(binary_path.read_text())
    binary_columns = ["binary_balanced_accuracy", "sensitivity", "specificity", "abnormal_auroc", "abnormal_average_precision"]
    display(pd.DataFrame(binary_report["test_reports"]).T.reindex(columns=binary_columns))
    primary_binary_jepa = binary_report["primary_jepa"]
    print("Validation-selected primary JEPA:", primary_binary_jepa)
    display(binary_report["paired_against_rf"][primary_binary_jepa])
    display(pd.DataFrame(binary_report["old_five_class_derived_reports"]).T.reindex(columns=binary_columns))
else:
    print("Direct-binary final test results remain sealed until WM_RELEASE_TEST_RESULTS=1.")

In [ ]:
lesson_evidence = persist_lesson("06_capstone_rf_vs_jepa", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.